# 3 · Moving house: editing relations

**The problem.** Alice moves. Her old home stays in the book, because her lodger still lives there, but her link
to it must go and a new one must appear. Her work address changes label. A new colleague joins who is "just like
Alice" apart from the name. Each of these is an ordinary edit, and each shows something about how the framework
treats relations as shared facts rather than private lists.

This case study reuses case study 2's address book. The schemas are repeated in one cell so the notebook stands on
its own.

In [1]:
from mbse_schemas.Framework import Proxies, Schemas
from toolkit import entries

Contact = (
    Schemas.OfObject.Builder()
    .properties(
        lambda prop: prop.name('given_name').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('family_name').of(lambda t: t.as_native(str)),
    )
    .create()
)
Address = (
    Schemas.OfObject.Builder()
    .properties(
        lambda prop: prop.name('street').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('locality').of(lambda t: t.as_native(str)),
    )
    .create()
)
ContactAddresses = (
    Schemas.OfRelation.Builder()
    .links('contact', 'address')
    .properties(lambda prop: prop.name('label').of(lambda t: t.as_native(str)))
    .create()
)
Contact = Schemas.OfObject.Builder(Contact).relations(lambda adj: adj.name('addresses').of(ContactAddresses).me('contact')).update()
Address = Schemas.OfObject.Builder(Address).relations(lambda adj: adj.name('residents').of(ContactAddresses).me('address')).update()
for name, schema in [('Contact', Contact), ('Address', Address), ('ContactAddresses', ContactAddresses)]:
    Proxies.register(name, schema)
Builders = Proxies.Builders


def show(obj, adjacency, link, field):
    return sorted((getattr(e[link], field), e.get('label', '-')) for e in entries(obj, adjacency))

In [2]:
old_home = Builders.Address().street('7 Old Road').locality('Oxford').create()
office = Builders.Address().street('1 Market Square').locality('Oxford').create()

alice = (
    Builders.Contact()
    .given_name('Alice')
    .addresses(lambda x: x.address(old_home).label('home'))
    .addresses(lambda x: x.address(office).label('work'))
    .create()
)
lodger = Builders.Contact().given_name('Lewis').addresses(lambda x: x.address(old_home).label('home')).create()

print('alice:   ', show(alice, 'addresses', 'address', 'street'))
print('old home:', show(old_home, 'residents', 'contact', 'given_name'))

alice:    [('1 Market Square', 'work'), ('7 Old Road', 'home')]
old home: [('Alice', 'home'), ('Lewis', 'home')]


## Removing an entry

There's no `alice.addresses.remove(...)`, for the same reason as in case study 2: an entry isn't owned by Alice.
You remove it through a builder, by visiting the adjacency and handing back the entry to drop. The builder's
`adjacency(name, callback)` gives you the adjacency handle `a`. `a.entries(...)` visits each entry, and
`a.remove(entry)` drops one. Nothing takes effect until `update()`:

In [3]:
def is_link_to(entry, link, target):
    found = []
    entry.link(link, lambda k: k.target(lambda t: found.append(t is target)))
    return found == [True]


(
    Builders.Contact(alice)
    .adjacency('addresses', lambda a: a.entries(lambda e: a.remove(e) if is_link_to(e, 'address', old_home) else None))
    .addresses(lambda x: x.address(lambda new: new.street('2 New Lane').locality('Bath')).label('home'))
    .update()
)

print('alice:   ', show(alice, 'addresses', 'address', 'street'))
print('old home:', show(old_home, 'residents', 'contact', 'given_name'))

alice:    [('1 Market Square', 'work'), ('2 New Lane', 'home')]
old home: [('Lewis', 'home')]


One `update()` removed the old fact and added the new one. Because an entry is one fact, it also left the old
address's `residents`. Lewis is still there, untouched, and the old address object still exists. Only the link went.

### Why the callback style for removal?

`a.entries(...)` hands you the entry *handles*, and `a.remove(...)` takes one back. You never get a list you could
keep, mutate and forget to write back. The builder decides what "remove" means, here dropping it from its working
copy until `update()`. A serializer or validator implementing the same protocol would do something else with the
same calls.

Filtering by what an entry *contains* is common enough that `toolkit.remove_entries` wraps it, using the same row
dicts as `entries`:

In [4]:
from toolkit import remove_entries

remove_entries(lodger, 'addresses', where=lambda row: row['address'] is old_home)
print('old home:', show(old_home, 'residents', 'contact', 'given_name'))

old home: []


## Changing an entry's property

An entry's properties are part of its identity: `(alice, office, 'work')` and `(alice, office, 'office')` are two
different facts, as you saw with duplicate elision. So "relabeling" is removing one fact and adding another, in one
builder:

In [5]:
(
    Builders.Contact(alice)
    .adjacency('addresses', lambda a: a.entries(lambda e: a.remove(e) if is_link_to(e, 'address', office) else None))
    .addresses(lambda x: x.address(office).label('office'))
    .update()
)
show(alice, 'addresses', 'address', 'street')

[('1 Market Square', 'office'), ('2 New Lane', 'home')]

## Cloning copies the relations too

A new colleague, Bob, works at the same office and lives at the same place as Alice. `clone()` copies properties
*and* entries, with the clone in the place of the original:

In [6]:
bob = Builders.Contact(alice).given_name('Bob').clone()

print('bob:   ', show(bob, 'addresses', 'address', 'street'))
print('office:', show(office, 'residents', 'contact', 'given_name'))

bob:    [('1 Market Square', 'office'), ('2 New Lane', 'home')]
office: [('Alice', 'office'), ('Bob', 'office')]


Notice what cloning means for the *other* end: the office now has two residents. That's the only consistent
reading of "a copy of Alice's facts with Bob in her place". Cloning a contact with no relations is just a copy of
its properties.

## What `update()` replaces

A builder started from an instance works on a **snapshot** of that instance, its properties and all of its entries.
`update()` writes the whole snapshot back: it drops every entry the object had and adds the builder's. That makes an
update all-or-nothing and easy to reason about. It also means two builders open on the same object at once don't
merge. The last `update()` wins:

In [7]:
first = Builders.Contact(bob).addresses(lambda x: x.address(old_home).label('weekend'))
second = Builders.Contact(bob).given_name('Robert')

first.update()
print('after first: ', show(bob, 'addresses', 'address', 'street'))
second.update()   # started before `first` was written, so it doesn't contain the weekend entry
print('after second:', show(bob, 'addresses', 'address', 'street'), bob.given_name)

after first:  [('1 Market Square', 'office'), ('2 New Lane', 'home'), ('7 Old Road', 'weekend')]
after second: [('1 Market Square', 'office'), ('2 New Lane', 'home')] Robert


**The rule:** open a builder, make the change, finish it. Don't keep builders around across other edits. (Whether
`update()` should instead merge is an open design question, recorded as finding F4 in the test plan. The behavior
above is what exists today.)

## Leaving an adjacency alone

The flip side: a builder that doesn't touch an adjacency writes back exactly the entries it read, so a
properties-only edit never loses relations:

In [8]:
Builders.Contact(alice).family_name('Liddell').update()
show(alice, 'addresses', 'address', 'street')

[('1 Market Square', 'office'), ('2 New Lane', 'home')]

## What you learned

- Entries are removed through a builder: `.adjacency(name, lambda a: a.entries(lambda e: a.remove(e) ...))`, then
  `update()`.
- An entry's properties are part of the fact, so changing one means remove and add.
- Removing an entry affects both ends, and never deletes the linked objects.
- `clone()` copies entries with the clone in place of the original, so other ends see the clone too.
- `update()` writes back the builder's whole snapshot. Keep builders short-lived.

**Next:** so far every relation linked two *different* kinds of object. Case study 4 models a family tree, where
people link to people, cycles appear, and "everything connected to X" becomes a question worth asking.